# Additional Feature Engineering

## Purpose

Validate and extend the order-level research table with price and product features.

## Rules and scope
- Keep one row per order_id and preserve all original values and labels.
- Preserve unknown values instead of filling them with zero.
- Price, item composition, seller assignments and product attributes are candidate predictors only if available at purchase. Historical product attributes are not versioned; this assumption must be documented.
- This version creates 12 additional columns: four price features, four product descriptors, and four quality or traceability indicators.
- Holiday and weather integration will follow; neither is implemented here.
- This notebook does not run hypothesis tests or train models.
- Export is disabled by default and never overwrites research_data.csv.


## 1 Load clean tables

Work on a copy. Rerunning this cell resets all derived features.

In [1]:
# Purpose: Locate the project root and load the required clean tables.

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd()

if (current_dir / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "data/processed/research_data.csv").exists():
    PROJECT_ROOT = current_dir.parent
else:
    raise FileNotFoundError(
        "Cannot find data/processed/research_data.csv."
    )

DATA_DIR = PROJECT_ROOT / "data/processed"

research = pd.read_csv(
    DATA_DIR / "research_data.csv",
    dtype={"order_id": "string", "customer_id": "string"},
)

items = pd.read_csv(
    DATA_DIR / "items_clean.csv",
    dtype={
        "order_id": "string",
        "product_id": "string",
        "seller_id": "string",
    },
)

products = pd.read_csv(
    DATA_DIR / "products_clean.csv",
    dtype={"product_id": "string"},
)

sellers = pd.read_csv(
    DATA_DIR / "sellers_clean.csv",
    dtype={
        "seller_id": "string",
        "seller_state": "string",
        "seller_zip_code_prefix": "string",
    },
)

# Work on a copy and preserve the original table.
enriched = research.copy()

display(pd.DataFrame([
    {"table": "research", "rows": len(research), "columns": research.shape[1]},
    {"table": "items", "rows": len(items), "columns": items.shape[1]},
    {"table": "products", "rows": len(products), "columns": products.shape[1]},
    {"table": "sellers", "rows": len(sellers), "columns": sellers.shape[1]},
]))

,table,rows,columns
0,research,99441,34
1,items,112650,7
2,products,32951,14
3,sellers,3095,4


## 2 Validate research structure

Check required columns, unique identifiers and valid labels.

In [2]:
# Purpose: Check order uniqueness and required price columns.

required_columns = [
    "order_id",
    "is_late",
    "items_value",
    "freight_value",
    "item_count",
]

missing_columns = [
    column for column in required_columns
    if column not in research.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

structure_checks = pd.DataFrame({
    "check": [
        "Total orders",
        "Missing order_id",
        "Blank order_id",
        "Rows in duplicated order_id groups",
        "Orders with valid target labels",
    ],
    "value": [
        len(research),
        research["order_id"].isna().sum(),
        research["order_id"].str.strip().eq("").sum(),
        research["order_id"].duplicated(keep=False).sum(),
        research["is_late"].isin([0, 1]).sum(),
    ],
})

display(structure_checks)

if (
    research["order_id"].isna().any()
    or research["order_id"].str.strip().eq("").any()
    or research["order_id"].duplicated().any()
):
    raise ValueError("Resolve order_id issues before continuing.")

,check,value
0,Total orders,99441
1,Missing order_id,0
2,Blank order_id,0
3,Rows in duplicated order_id groups,0
4,Orders with valid target labels,96470


## 3 Check price inputs

Preserve missing values. Zero freight is not automatically an error.

In [3]:
# Purpose: Detect missing, invalid, and negative price inputs.

price_columns = ["items_value", "freight_value", "item_count"]

price_checks = []

for column in price_columns:
    original = enriched[column]
    numeric = pd.to_numeric(original, errors="coerce")

    price_checks.append({
        "feature": column,
        "missing_rows": original.isna().sum(),
        "invalid_numeric_rows": (
            original.notna() & numeric.isna()
        ).sum(),
        "infinite_rows": numeric.isin([np.inf, -np.inf]).sum(),
        "negative_rows": numeric.lt(0).sum(),
        "zero_rows": numeric.eq(0).sum(),
    })

    enriched[column] = numeric

price_checks = pd.DataFrame(price_checks)

display(price_checks)

,feature,missing_rows,invalid_numeric_rows,infinite_rows,negative_rows,zero_rows
0,items_value,775,0,0,0,0
1,freight_value,775,0,0,0,338
2,item_count,775,0,0,0,0


## 4 Reconcile totals

Verify totals and missing patterns against item records.

In [4]:
# Purpose: Recalculate order totals and verify the research table.

item_values = items.copy()

for column in ["price", "freight_value"]:
    item_values[column] = pd.to_numeric(
        item_values[column], errors="raise"
    )

item_totals = (
    item_values.groupby("order_id")
    .agg(
        expected_items_value=(
            "price", lambda values: values.sum(min_count=1)
        ),
        expected_freight_value=(
            "freight_value", lambda values: values.sum(min_count=1)
        ),
        expected_item_count=("order_id", "size"),
    )
    .reset_index()
)

comparison = enriched.merge(
    item_totals,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

checks = []

for column in ["items_value", "freight_value", "item_count"]:
    actual = comparison[column]
    expected = comparison[f"expected_{column}"]

    comparable = actual.notna() & expected.notna()

    checks.append({
        "feature": column,
        "missing_pattern_mismatches": (
            actual.isna() != expected.isna()
        ).sum(),
        "comparable_rows": comparable.sum(),
        "value_mismatches": (
            ~np.isclose(
                actual[comparable],
                expected[comparable],
                rtol=0,
                atol=1e-6,
            )
        ).sum(),
    })

total_checks = pd.DataFrame(checks)

display(total_checks)

print(
    "Orders without item records:",
    comparison["_merge"].eq("left_only").sum(),
)

,feature,missing_pattern_mismatches,comparable_rows,value_mismatches
0,items_value,0,98666,0
1,freight_value,0,98666,0
2,item_count,0,98666,0


Orders without item records: 775


## 5 Create price features

order_total_value: BRL; freight_to_items_ratio: unitless; mean_item_value and freight_per_item: BRL per item row. These are transformations of existing information.

In [5]:
# Purpose: Create price features without replacing missing inputs.

if total_checks[
    ["missing_pattern_mismatches", "value_mismatches"]
].to_numpy().sum() != 0:
    raise ValueError("Resolve total mismatches before creating features.")

enriched["order_total_value"] = (
    enriched["items_value"] + enriched["freight_value"]
)

enriched["freight_to_items_ratio"] = (
    enriched["freight_value"]
    / enriched["items_value"].where(enriched["items_value"] > 0)
)

enriched["mean_item_value"] = (
    enriched["items_value"]
    / enriched["item_count"].where(enriched["item_count"] > 0)
)

enriched["freight_per_item"] = (
    enriched["freight_value"]
    / enriched["item_count"].where(enriched["item_count"] > 0)
)

new_price_features = [
    "order_total_value",
    "freight_to_items_ratio",
    "mean_item_value",
    "freight_per_item",
]

display(enriched[
    ["order_id", "items_value", "freight_value", "item_count"]
    + new_price_features
].head())

display(pd.DataFrame({
    "missing_rows": enriched[new_price_features].isna().sum(),
    "infinite_rows": enriched[new_price_features]
        .isin([np.inf, -np.inf]).sum(),
}))

,order_id,items_value,freight_value,item_count,order_total_value,freight_to_items_ratio,mean_item_value,freight_per_item
0,e481f51cbdc54678b7cc49136f2d6af7,29.99,8.72,1.0,38.71,0.290764,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,118.70,22.76,1.0,141.46,0.191744,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,159.90,19.22,1.0,179.12,0.120200,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,45.00,27.20,1.0,72.20,0.604444,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,19.90,8.72,1.0,28.62,0.438191,19.90,8.72


,missing_rows,infinite_rows
order_total_value,775,0
freight_to_items_ratio,775,0
mean_item_value,775,0
freight_per_item,775,0


## 6 Attach product attributes

Use a many-to-one merge to prevent item duplication.

In [6]:
# Purpose: Attach product attributes without multiplying item rows.

product_columns = [
    "product_id",
    "product_weight_g_valid",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_category_name",
]

missing_columns = [
    column for column in product_columns
    if column not in products.columns
]

if missing_columns:
    raise ValueError(f"Missing product columns: {missing_columns}")

if (
    products["product_id"].isna().any()
    or products["product_id"].duplicated().any()
):
    raise ValueError("Product IDs must be present and unique.")

item_products = items.merge(
    products[product_columns],
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="_product_match",
)

display(pd.DataFrame({
    "check": [
        "Item rows before merge",
        "Item rows after merge",
        "Items without a matching product",
    ],
    "value": [
        len(items),
        len(item_products),
        item_products["_product_match"].eq("left_only").sum(),
    ],
}))

,check,value
0,Item rows before merge,112650
1,Item rows after merge,112650
2,Items without a matching product,0


## 7 Validate measurements

Keep positive finite weight and dimensions. Product volume is not parcel volume.

In [7]:
# Purpose: Calculate product volume and identify usable measurements.

measurement_columns = [
    "product_weight_g_valid",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for column in measurement_columns:
    item_products[column] = pd.to_numeric(
        item_products[column], errors="raise"
    )

weight = item_products["product_weight_g_valid"]

item_products["item_weight_g"] = weight.where(
    weight.gt(0) & np.isfinite(weight)
)

dimension_columns = [
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

dimensions = item_products[dimension_columns]

valid_dimensions = (
    dimensions.notna().all(axis=1)
    & dimensions.gt(0).all(axis=1)
    & np.isfinite(dimensions).all(axis=1)
)

item_products["item_volume_cm3"] = (
    dimensions.prod(axis=1, min_count=3)
    .where(valid_dimensions)
)

item_products["product_category_name"] = (
    item_products["product_category_name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

display(pd.DataFrame({
    "missing_rows": item_products[
        ["item_weight_g", "item_volume_cm3", "product_category_name"]
    ].isna().sum()
}))

,missing_rows
item_weight_g,26
item_volume_cm3,18
product_category_name,1603


## 8 Aggregate product features

Create total_product_weight_g, total_product_volume_cm3 and product_category_count. Require full coverage for each measure. Coverage ratios are quality indicators, not automatically approved predictors.

In [8]:
# Purpose: Aggregate product features to one row per order.

product_features = (
    item_products.groupby("order_id")
    .agg(
        product_item_rows=("order_id", "size"),
        weight_known_item_count=("item_weight_g", "count"),
        volume_known_item_count=("item_volume_cm3", "count"),
        category_known_item_count=("product_category_name", "count"),
        total_product_weight_g=(
            "item_weight_g", lambda values: values.sum(min_count=1)
        ),
        total_product_volume_cm3=(
            "item_volume_cm3", lambda values: values.sum(min_count=1)
        ),
        product_category_count=("product_category_name", "nunique"),
    )
    .reset_index()
)

for prefix in ["weight", "volume", "category"]:
    product_features[f"{prefix}_coverage_ratio"] = (
        product_features[f"{prefix}_known_item_count"]
        / product_features["product_item_rows"]
    )

# Require complete information for order-level totals and category counts.
for feature, coverage in [
    ("total_product_weight_g", "weight_coverage_ratio"),
    ("total_product_volume_cm3", "volume_coverage_ratio"),
    ("product_category_count", "category_coverage_ratio"),
]:
    product_features[feature] = product_features[feature].where(
        product_features[coverage].eq(1)
    )

new_product_columns = [
    "total_product_weight_g",
    "total_product_volume_cm3",
    "product_category_count",
    "weight_coverage_ratio",
    "volume_coverage_ratio",
    "category_coverage_ratio",
]

# Remove previous copies so this cell can be rerun safely.
enriched = enriched.drop(
    columns=new_product_columns, errors="ignore"
).merge(
    product_features[["order_id"] + new_product_columns],
    on="order_id",
    how="left",
    validate="one_to_one",
)

assert len(enriched) == len(research)
assert enriched["order_id"].is_unique

pd.testing.assert_series_equal(
    enriched.set_index("order_id")["is_late"].sort_index(),
    research.set_index("order_id")["is_late"].sort_index(),
)

display(pd.DataFrame({
    "missing_rows": enriched[new_product_columns].isna().sum(),
    "missing_percent": (
        enriched[new_product_columns].isna().mean() * 100
    ).round(2),
}))

,missing_rows,missing_percent
total_product_weight_g,797,0.80
total_product_volume_cm3,791,0.80
product_category_count,2226,2.24
weight_coverage_ratio,775,0.78
volume_coverage_ratio,775,0.78
category_coverage_ratio,775,0.78


## 9 Select dominant category

Choose the category with the largest total item value. Require complete categories and valid prices; break ties alphabetically. dominant_category_tied is a traceability flag.

In [9]:
# Purpose: Select the category with the highest item value per order.

category_items = item_products.copy()

category_items["price"] = pd.to_numeric(
    category_items["price"], errors="raise"
)

category_items["valid_category_price"] = (
    category_items["price"].notna()
    & np.isfinite(category_items["price"])
    & category_items["price"].ge(0)
)

category_checks = (
    category_items.groupby("order_id")
    .agg(
        all_categories_known=(
            "product_category_name",
            lambda values: values.notna().all(),
        ),
        all_prices_valid=("valid_category_price", "all"),
    )
)

eligible_category_orders = category_checks.index[
    category_checks["all_categories_known"]
    & category_checks["all_prices_valid"]
]

category_values = (
    category_items[
        category_items["order_id"].isin(eligible_category_orders)
    ]
    .groupby(["order_id", "product_category_name"], as_index=False)
    .agg(category_items_value=("price", "sum"))
)

max_value = category_values.groupby("order_id")[
    "category_items_value"
].transform("max")

top_categories = category_values[
    np.isclose(
        category_values["category_items_value"],
        max_value,
        rtol=0,
        atol=1e-6,
    )
].copy()

top_categories["dominant_category_tied"] = (
    top_categories.groupby("order_id")["product_category_name"]
    .transform("size")
    .gt(1)
    .astype("Int64")
)

dominant_categories = (
    top_categories
    .sort_values(["order_id", "product_category_name"])
    .drop_duplicates("order_id")
    .rename(columns={
        "product_category_name": "dominant_product_category"
    })
)

category_columns = [
    "dominant_product_category",
    "dominant_category_tied",
]

enriched = enriched.drop(
    columns=category_columns, errors="ignore"
).merge(
    dominant_categories[["order_id"] + category_columns],
    on="order_id",
    how="left",
    validate="one_to_one",
)

enriched["dominant_category_tied"] = (
    enriched["dominant_category_tied"].astype("Int64")
)

assert len(enriched) == len(research)
assert enriched["order_id"].is_unique

display(pd.DataFrame({
    "missing_rows": enriched[category_columns].isna().sum(),
    "missing_percent": (
        enriched[category_columns].isna().mean() * 100
    ).round(2),
}))

print(
    "Orders with tied dominant categories:",
    enriched["dominant_category_tied"].eq(1).sum(),
)

,missing_rows,missing_percent
dominant_product_category,2226,2.24
dominant_category_tied,2226,2.24


Orders with tied dominant categories: 42


## 10 Final validation

Verify all original columns, row counts, target labels and feature coverage before continuing.

In [10]:
# Purpose: Validate the complete price and product feature set.
additional_features = new_price_features + new_product_columns + category_columns
assert len(additional_features) == len(set(additional_features)) == 12
assert len(enriched) == len(research)
assert enriched["order_id"].notna().all()
assert enriched["order_id"].is_unique
assert enriched.shape[1] == research.shape[1] + 12
pd.testing.assert_frame_equal(
    enriched.set_index("order_id")[research.columns.drop("order_id")].sort_index(),
    research.set_index("order_id").sort_index(),
    check_dtype=False,
)
numeric_features = [c for c in additional_features if c != "dominant_product_category"]
assert not enriched[numeric_features].isin([np.inf, -np.inf]).any().any()
for column in ["weight_coverage_ratio", "volume_coverage_ratio", "category_coverage_ratio"]:
    assert enriched[column].dropna().between(0, 1).all()
display(pd.DataFrame({
    "missing_rows": enriched[additional_features].isna().sum(),
    "missing_percent": (enriched[additional_features].isna().mean() * 100).round(2),
}))
print("All final checks passed.")
print("Enriched dataset shape:", enriched.shape)


,missing_rows,missing_percent
order_total_value,775,0.78
freight_to_items_ratio,775,0.78
mean_item_value,775,0.78
freight_per_item,775,0.78
total_product_weight_g,797,0.80
total_product_volume_cm3,791,0.80
product_category_count,2226,2.24
weight_coverage_ratio,775,0.78
volume_coverage_ratio,775,0.78
category_coverage_ratio,775,0.78


All final checks passed.
Enriched dataset shape: (99441, 46)


## 11 Optional export

Leave export disabled until external data integration is complete.

In [12]:
# Purpose: Save the validated price and product dataset.

output_path = DATA_DIR / "research_data_price_product.csv"

enriched.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Dataset shape:", enriched.shape)

Saved: c:\Users\LENOVO\Downloads\Group5_STAT3013.R11-main (1)\Group5_STAT3013.R11-main\data\processed\research_data_price_product.csv
Dataset shape: (99441, 46)
